In [130]:
# Loading the data
import json
import pandas as pd
from pathlib import Path
import sys
import matplotlib.pyplot as plt
from sklearn.covariance import MinCovDet
import numpy as np

# Define the project root (one folder up from /workshop)
project_root = Path("..").resolve()

# Add the project root to sys.path so Jupyter can find your 'src' module
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# Load data/valencia_cf_1/matches.json
matches_path = project_root / "tests" / "fixtures" / "testing_data" / "data" / "valencia_cf_1" / "matches.json"
raw_matches = json.load(open(matches_path))
print(f"Loaded {len(raw_matches)} matches from {matches_path}")

Loaded 100 matches from C:\Users\kazik\projects\Gaffers-Clipboard\tests\fixtures\testing_data\data\valencia_cf_1\matches.json


In [131]:
normalized_df = pd.json_normalize(
    raw_matches,
    record_path="player_performances",
    meta=[
        "id", ["data", "half_length"], 
        ["data", "home_team_name"], 
        ["data", "away_team_name"], 
        ["data", "home_stats", "possession"], 
        ["data", "away_stats", "possession"],
        ["data", "home_stats", "xg"],
        ["data", "away_stats", "xg"],
    ]
)

normalized_df["team_possession"] = float("nan")
normalized_df["team_xg"] = float("nan")
normalized_df["opponent_xg"] = float("nan")

for idx, row in normalized_df.iterrows():
    if row["data.home_team_name"] == "Valencia CF":
        normalized_df.at[idx, "team_possession"] = row["data.home_stats.possession"]
        normalized_df.at[idx, "team_xg"] = row["data.home_stats.xg"]
        normalized_df.at[idx, "opponent_xg"] = row["data.away_stats.xg"]
    else:
        normalized_df.at[idx, "team_possession"] = row["data.away_stats.possession"]
        normalized_df.at[idx, "team_xg"] = row["data.away_stats.xg"]
        normalized_df.at[idx, "opponent_xg"] = row["data.home_stats.xg"]

normalized_df = normalized_df.drop(columns=[
    "data.home_team_name", "data.away_team_name", "data.home_stats.possession", "data.away_stats.possession", "data.home_stats.xg", "data.away_stats.xg"])

normalized_df = normalized_df.rename(columns={
    "data.half_length": "half_length",
    "id": "match_id",
})

normalized_df = normalized_df[normalized_df["performance_type"] != "GK"]

normalized_df = normalized_df.dropna(axis=1, how="all")

pos_df = normalized_df[normalized_df["positions_played"].apply(lambda x: x == ["ST"])]

pos_df = pos_df.dropna(axis=1, how="all")

pos_df = pos_df.drop(columns=["performance_type", "positions_played"])

cols = pos_df.columns.tolist()
cols.insert(0, cols.pop(cols.index("match_id")))
pos_df = pos_df[cols]

print(f"Number of rows: {pos_df.shape[0]}")
print(f"Number of columns: {pos_df.shape[1]}")
print("Columns:")
for col in cols:
    print(f" - {col}")

# Output the max and min of every column, to get a sense of the range of values
for col in pos_df.columns:
    if col in ["match_id", "player_id"]:
        continue
    if pos_df[col].dtype in ["int64", "float64"]:
        print(f"{col}: min={pos_df[col].min()}, max={pos_df[col].max()}")

Number of rows: 151
Number of columns: 23
Columns:
 - match_id
 - player_id
 - goals
 - assists
 - shots
 - shot_accuracy
 - passes
 - pass_accuracy
 - dribbles
 - dribble_success_rate
 - tackles
 - tackle_success_rate
 - offsides
 - fouls_committed
 - possession_won
 - possession_lost
 - minutes_played
 - distance_covered
 - distance_sprinted
 - half_length
 - team_possession
 - team_xg
 - opponent_xg
goals: min=0.0, max=5.0
assists: min=0.0, max=2.0
shots: min=0.0, max=10.0
shot_accuracy: min=0.0, max=100.0
passes: min=0.0, max=31.0
pass_accuracy: min=0.0, max=100.0
dribbles: min=0.0, max=30.0
dribble_success_rate: min=0.0, max=100.0
tackles: min=0.0, max=6.0
tackle_success_rate: min=0.0, max=100.0
offsides: min=0.0, max=2.0
fouls_committed: min=0.0, max=2.0
possession_won: min=0.0, max=4.0
possession_lost: min=0.0, max=12.0
minutes_played: min=4.0, max=93.0
distance_covered: min=0.4, max=13.3
distance_sprinted: min=0.1, max=5.4
team_possession: min=39.0, max=71.0
team_xg: min=0.7, m

In [132]:
# 1. Select every column EXCEPT your identifiers
cols_to_convert = pos_df.columns.drop(['match_id', 'player_id'])

# 2. Force them all to numeric in one swoop
pos_df[cols_to_convert] = pos_df[cols_to_convert].apply(pd.to_numeric, errors='coerce')

# Optional: If any weird JSON text like "N/A" got turned into a NaN, 
# you can safely fill them with 0s to keep the math from breaking later.
pos_df[cols_to_convert] = pos_df[cols_to_convert].fillna(0)

In [133]:
# Step 1: Removing players with less than 10 minutes played
pos_df = pos_df[pos_df["minutes_played"] >= 10]

In [134]:
# Step 1: Volume masking
# For each percentage stat, their associated volume attempted stat must reach a certain threshold for the percentage 
# stat to be considered valid. For example, if a player attempted 0 passes, their pass accuracy should not be considered valid,
# and should be set to the mean.
volume_perc_pairs = [
    ("passes", "pass_accuracy"),
    ("shots", "shot_accuracy"),
    ("dribbles", "dribble_success_rate"),
    ("tackles", "tackle_success_rate")
]
volume_masks = {
    "passes": 5,
    "shots": 2,
    "dribbles": 3,
    "tackles": 2
}

for volume_col, perc_col in volume_perc_pairs:
    # apply the mask
    valid = pos_df[pos_df[volume_col] >= volume_masks[volume_col]]
    mean_value = valid[perc_col].mean()
    pos_df[perc_col] = pos_df.apply(
        lambda r: r[perc_col] if r[volume_col] >= volume_masks[volume_col] else mean_value,
        axis=1
    )

In [135]:
# ---------------------------------------------------------
# Step 1: Standardize Absolute Volume (The Half-Length Fix)
# ---------------------------------------------------------
cum_columns = ["goals", "assists", "shots", "passes", "dribbles", "tackles", 
               "possession_won", "possession_lost", "fouls_committed", "offsides", 
               "distance_covered", "distance_sprinted"]

h_base = 10.0

for col in cum_columns:
    # We overwrite the raw stat to represent a standard 10-minute half game.
    # This ensures 5-min half games don't corrupt our dataset averages later.
    pos_df[col] = pos_df[col] * (h_base / pos_df["half_length"])


# ---------------------------------------------------------
# Step 2: Set Bayesian Smoothing Parameters
# ---------------------------------------------------------
dummy_minutes = 30.0  # Shrinkage anchor (requires ~a half of football to break away from mean)


# ---------------------------------------------------------
# Step 3: Apply Smoothed Per-90 Scaling
# ---------------------------------------------------------
# A. Rare Stats (Assume 0 for dummy minutes)
rare_cols = ["goals", "assists", "shots"]
for col in rare_cols:
    pos_df[f"{col}_p90"] = (pos_df[col] / (pos_df["minutes_played"] + dummy_minutes)) * 90.0


# B. Volume Stats (Assume dataset average for dummy minutes)
volume_cols = ["passes", "dribbles", "tackles", "possession_won", "possession_lost", 
               "fouls_committed", "offsides", "distance_covered", "distance_sprinted"]

for col in volume_cols:
    # 1. Calculate the true, half-length-adjusted dataset average Per 90
    league_avg_p90 = (pos_df[col].sum() / pos_df["minutes_played"].sum()) * 90.0
    
    # 2. Calculate the dummy stat allocation for 45 minutes
    dummy_stat = league_avg_p90 * (dummy_minutes / 90.0)
    
    # 3. Apply the final smoothed formula
    pos_df[f"{col}_p90"] = ((pos_df[col] + dummy_stat) / (pos_df["minutes_played"] + dummy_minutes)) * 90.0

In [136]:
pos_df["xT_bonus_p90"] = 0.25 * (pos_df["distance_sprinted_p90"] / pos_df["distance_covered_p90"]) * np.log(pos_df["pass_accuracy"] * pos_df["passes_p90"] + 1)

In [137]:
# Match supremacy scalar
gamma = 0.2
delta_xg = (pos_df['team_xg'] + 1) / (pos_df['opponent_xg'] + 1)
match_supremacy_scalar = gamma * np.log(delta_xg)

In [138]:
# Now we import weights from root/workshop/position_weights.json
# Which is in the format {"CB": {"goals_p90": 0.3, ...}, "CM": {...}, "ST": {...}}
col_names = ["goals_p90", "assists_p90", "shots_p90", "shot_accuracy", "passes_p90", "pass_accuracy",
              "dribbles_p90", "dribble_success_rate", "tackles_p90", "tackle_success_rate", "offsides_p90",
              "fouls_committed_p90", "possession_won_p90", "possession_lost_p90", "distance_covered_p90", "distance_sprinted_p90", "xT_bonus_p90"]
position_weights_path = project_root / "workshop" / "position_weights.json"
with open(position_weights_path, "r") as f:
    position_weights = json.load(f)
weights_dict = position_weights["ST"]
final_weights = np.array([weights_dict[col] for col in col_names])

In [139]:
# Now we import the means and stds from root/workshop/position_means_stds.json
# Which is in the format {"CB": {"goals_p90": {"mean": 0.1, "std": 0.2}, ...}, "CM": {...}, "ST": {...}}
position_means_stds_path = project_root / "workshop" / "position_means_stds.json"
with open(position_means_stds_path, "r") as f:
    position_means_stds = json.load(f)
means_stds_dict = position_means_stds["ST"]

In [140]:
# Compute final z-scores
negative_stats = ["fouls_committed_p90", "possession_lost_p90", "offsides_p90"]
for col in col_names:
    mean = means_stds_dict[col]["mean"]
    std = means_stds_dict[col]["std"]
    if std == 0:
        pos_df[f"{col}_z"] = 0
    elif col in negative_stats:
        pos_df[f"{col}_z"] = (mean - pos_df[col]) / std
    else:
        pos_df[f"{col}_z"] = (pos_df[col] - mean) / std

In [141]:
# 1. The Ghosting Floor (Limit the bleeding, but let them be punished)
# -1.0 means they will lose a full rating point for doing absolutely nothing, 
# but it stops them from getting a 2.0/10 rating just because they didn't shoot.
ghosting_stats = ['goals_p90_z', 'assists_p90_z', 'shots_p90_z']
for col in ghosting_stats:
    pos_df[col] = np.clip(pos_df[col], a_min=-1.0, a_max=None)

# 2. Striker Volatility Floor
# Strikers operate in high-traffic areas. 0% accuracy shouldn't ruin a game if they ran 11km.
efficiency_stats = ['shot_accuracy_z', 'pass_accuracy_z', 'dribble_success_rate_z']
for col in efficiency_stats:
    pos_df[col] = np.clip(pos_df[col], a_min=-0.75, a_max=None)

# 3. Cap the Offsides/Discipline Penalty
# If a striker gets caught offside 5 times, it should hurt (e.g., dropping them an 8.5 to a 7.9),
# but it shouldn't mathematically erase a 2-goal performance. 
pos_df["offsides_p90_z"] = np.clip(pos_df["offsides_p90_z"], a_min=-1.5, a_max=None)

In [142]:
z_score_cols = [f"{col}_z" for col in col_names]
z_matrix = pos_df[z_score_cols].values
pos_df['raw_score'] = np.dot(z_matrix, final_weights)

In [143]:
# Match Winners
pos_df['raw_score'] += np.where(pos_df['goals_p90_z'] > 0, pos_df['goals_p90_z'] * 1.2, 0)
pos_df['raw_score'] += np.where(pos_df['assists_p90_z'] > 0, pos_df['assists_p90_z'] * 0.6, 0)

# The Target Man / Complete Forward Bonus
# If they drop deep and put on a passing/carrying masterclass, give them a secondary boost
pos_df['raw_score'] += np.where(pos_df['passes_p90_z'] > 2, (pos_df['passes_p90_z'] - 2) * 0.2, 0)
pos_df['raw_score'] += np.where(pos_df['dribbles_p90_z'] > 2, (pos_df['dribbles_p90_z'] - 2) * 0.2, 0)

In [144]:
# ---------------------------------------------------------
# The Black Hole Penalty (Failed Hold-up Play)
# ---------------------------------------------------------
# 1. Define what a "Positive Involvement" is for a Striker
positive_involvements = pos_df['passes'] + pos_df['dribbles'] + pos_df['shots']

# 2. Prevent division by zero if a player literally did nothing
safe_positive_involvements = np.maximum(positive_involvements, 1)

# 3. Calculate the "Turnover Ratio"
# How many times did they lose it compared to doing something good?
turnover_ratio = pos_df['possession_lost'] / safe_positive_involvements

# 4. The Trigger Mask
# They must have lost the ball a decent amount (> 4 times) AND 
# their turnover ratio must be disastrous (e.g., losing it 1.5x more often than keeping it)
black_hole_mask = (pos_df['possession_lost'] > 4) & (turnover_ratio > 1.5)

# 5. The Proportional Penalty
# For every possession lost ABOVE their positive involvements, we dock them 0.08 raw score points
excess_losses = np.maximum(0, pos_df['possession_lost'] - positive_involvements)
black_hole_penalty = np.where(black_hole_mask, excess_losses * 0.08, 0)

# Cap the bleeding at -0.6 so it doesn't break the 1-10 scale
pos_df['raw_score'] -= np.clip(black_hole_penalty, a_min=0.0, a_max=0.6)

In [145]:
# ---------------------------------------------------------
# The Elite Hold-Up Bonus (The Anti-Black Hole)
# ---------------------------------------------------------
# 1. Calculate positive involvements and safe losses
positive_involvements = pos_df['passes'] + pos_df['dribbles'] + pos_df['shots']
safe_losses = np.maximum(pos_df['possession_lost'], 1)

# 2. Check if they were heavily involved but rarely dispossessed
# An elite ratio is keeping it 4+ times for every 1 loss, with at least 15 touches
holdup_mask = (positive_involvements >= 15) & ((positive_involvements / safe_losses) >= 4.0)

# 3. Calculate "Excess Retention"
# We assume a normal striker loses it 1 time for every 3 touches. 
# We calculate how many positive touches they had ABOVE that normal threshold.
normal_expected_touches = pos_df['possession_lost'] * 3
excess_retention = np.maximum(0, positive_involvements - normal_expected_touches)

# 4. The Proportional Reward
# Give +0.02 raw score points for every extra time they safely kept the ball
holdup_bonus = np.where(holdup_mask, excess_retention * 0.02, 0)

# Cap the maximum bonus at +0.4 so it doesn't rival actually scoring a goal
pos_df['raw_score'] += np.clip(holdup_bonus, a_min=0.0, a_max=0.4)

In [146]:
# ---------------------------------------------------------
# The Advanced Wasteful Finisher Penalty (SV-xG Model)
# ---------------------------------------------------------
# 1. Estimate their personal Expected Goals (assume 0.15 xG per shot)
pos_df['estimated_xg'] = pos_df['shots'] * 0.15

# 2. Calculate their Underperformance (Did they score fewer than expected?)
pos_df['finishing_deficit'] = pos_df['estimated_xg'] - pos_df['goals']

# 3. The Trigger Mask
# We only punish them if they took a decent volume of shots (> 3) AND 
# they underperformed their expected goals by more than 0.75 (roughly 5+ missed chances).
wasteful_mask = (pos_df['shots'] > 3) & (pos_df['finishing_deficit'] > 0.75)

# 4. The Proportional Penalty
# For every 1.0 expected goal they wasted, dock them 0.25 raw score points.
wasteful_penalty = np.where(wasteful_mask, pos_df['finishing_deficit'] * 0.25, 0)

# Apply the penalty, capping the maximum bleed at -0.8 to protect the 1-10 scale
pos_df['raw_score'] -= np.clip(wasteful_penalty, a_min=0.0, a_max=0.8)

In [147]:
k = 0.85
s_0 = np.log(2/3) / k
# Create a new column, applying the sigmoid transformation to the raw score
pos_df['raw_rating'] = 10 * (1 / (1 + np.exp(-k * (pos_df['raw_score'] - s_0))))

In [148]:
pos_df['final_rating'] = pos_df['raw_rating'] - match_supremacy_scalar

In [149]:
pos_df['final_rating'] = pos_df['final_rating'].apply(lambda x: max(0, min(10, x)))

In [150]:
random_row = pos_df.sample(n=1).iloc[0]
for col in pos_df.columns:
    print(f"{col}: {random_row[col]:.2f},")

match_id: 62.00,
player_id: 65.00,
goals: 1.00,
assists: 0.00,
shots: 2.00,
shot_accuracy: 100.00,
passes: 2.00,
pass_accuracy: 85.38,
dribbles: 4.00,
dribble_success_rate: 50.00,
tackles: 1.00,
tackle_success_rate: 20.91,
offsides: 0.00,
fouls_committed: 0.00,
possession_won: 0.00,
possession_lost: 2.00,
minutes_played: 13.00,
distance_covered: 1.70,
distance_sprinted: 0.60,
half_length: 10.00,
team_possession: 66.00,
team_xg: 10.70,
opponent_xg: 0.30,
goals_p90: 2.09,
assists_p90: 0.00,
shots_p90: 4.19,
passes_p90: 13.46,
dribbles_p90: 17.63,
tackles_p90: 3.21,
possession_won_p90: 0.62,
possession_lost_p90: 7.94,
fouls_committed_p90: 0.04,
offsides_p90: 0.22,
distance_covered_p90: 10.78,
distance_sprinted_p90: 2.93,
xT_bonus_p90: 0.48,
goals_p90_z: 1.86,
assists_p90_z: -0.40,
shots_p90_z: 1.01,
shot_accuracy_z: 1.66,
passes_p90_z: 0.23,
pass_accuracy_z: -0.00,
dribbles_p90_z: 1.26,
dribble_success_rate_z: -0.75,
tackles_p90_z: 1.42,
tackle_success_rate_z: 0.00,
offsides_p90_z: 0.21,
